# Auditoría y recopilación — primera ronda de LLM


## Qué revisa

- Los dos archivos compartidos y sus huellas SHA256.
- Presencia de resultados de los 22 modelos previstos y de los seis casos por modelo.
- Diferencias entre configuraciones, modos solicitados y razonamiento observado.
- Duplicados idénticos, versiones conflictivas de un mismo `execution_id`, rescates y reintentos.
- Coincidencia literal de los mensajes con V3 y metadatos faltantes de registros antiguos.
- Terminaciones, bloques de razonamiento incrustados y texto final recuperable.


In [1]:
# Auditoría local: no consulta Ollama ni genera nuevas respuestas.
import json
import re
import hashlib
import zipfile
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime, timezone

# Déjalo en None si la libreta está en 05_Pruebas/00_Evaluacion.
# Solo es necesario escribir una ruta si la detección automática falla.
RAIZ_MANUAL = None
NOMBRE_SALIDA = "resultados_llm_ronda1.zip"
SHA_CASOS_ESPERADO = "f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d"
SHA_CONFIG_ESPERADO = "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"

MODELOS_ESPERADOS = [
    "qwen2.5:7b",
    "ministral-3:8b-instruct-2512-q4_K_M",
    "granite4.2:8b-q4_K_M",
    "qwen3.5:9b-q4_K_M",
    "gemma4:12b-it-q4_K_M",
    "phi4-mini:3.8b-q4_K_M",
    "granite4.2:3b-q4_K_M",
    "qwen3.5:4b-q4_K_M",
    "llama3.1:8b-instruct-q4_K_M",
    "deepseek-r1:8b-0528-qwen3-q4_K_M",
    "phi4:14b-q4_K_M",
    "ministral-3:14b-instruct-2512-q4_K_M",
    "qwen2.5:14b-instruct-q4_K_M",
    "command-r7b:7b-12-2024-q4_K_M",
    "qwen3.8:27b-q4_K_M",
    "gemma4:26b-a4b-it-qat",
    "gpt-oss:20b",
    "mistral-small3.2:24b-instruct-2506-q4_K_M",
    "qwen3:30b-a3b-instruct-2507-q4_K_M",
    "gemma4:31b-it-qat",
    "granite4.2:30b-q4_K_M",
    "lfm2.5:8b-a1b-q4_K_M",
]


def huella_bytes(datos):
    return hashlib.sha256(datos).hexdigest()


def serializar(valor):
    return json.dumps(valor, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def localizar_raiz():
    if RAIZ_MANUAL is not None:
        candidatos = [Path(RAIZ_MANUAL).expanduser().resolve()]
    else:
        actual = Path.cwd().resolve()
        candidatos = [actual, actual.parent, actual / "05_Pruebas"]
        # Permite que VS Code sitúe el kernel en otra carpeta del repositorio.
        candidatos += [p / "05_Pruebas" for p in list(actual.parents)[:3]]
    for candidato in dict.fromkeys(candidatos):
        if all((candidato / n).is_file() for n in
               ("casos_comparacion_v1.json", "configuracion_v3.json")):
            return candidato
    raise FileNotFoundError(
        "No se encontró la carpeta con los dos JSON compartidos. "
        "Coloca la libreta en 05_Pruebas/00_Evaluacion o indica RAIZ_MANUAL."
    )


RAIZ = localizar_raiz()
SALIDA = RAIZ / "00_Evaluacion"
comunes = {}
for nombre in ("casos_comparacion_v1.json", "configuracion_v3.json"):
    comunes[nombre] = (RAIZ / nombre).read_bytes()

sha_casos = huella_bytes(comunes["casos_comparacion_v1.json"])
sha_config = huella_bytes(comunes["configuracion_v3.json"])
if sha_casos != SHA_CASOS_ESPERADO or sha_config != SHA_CONFIG_ESPERADO:
    raise ValueError(
        "Los JSON compartidos no coinciden con las huellas de la ronda V3. "
        "No se ha modificado ningún archivo. Revisa qué versiones estás usando."
    )
casos = json.loads(comunes["casos_comparacion_v1.json"].decode("utf-8-sig"))
config = json.loads(comunes["configuracion_v3.json"].decode("utf-8-sig"))
casos_por_id = {c["case_id"]: c for c in casos}
ids_esperados = set(casos_por_id)
if config.get("cases_sha256") != sha_casos or config.get("prompt_version") != "V3":
    raise ValueError("Configuración común incompatible con los casos de esta ronda.")

# Una fuente puede contener copias de ejecuciones ya presentes en otra fuente.
# Se preservan los archivos completos; la auditoría no cuenta dos veces una
# ejecución con el mismo execution_id y exactamente el mismo contenido.
fuentes = []
archivos_bytes = {}
incidencias = []
registros = {}
id_a_huellas = defaultdict(set)
copias_identicas = 0
EXCLUIR = {"00_Evaluacion", ".git", ".ipynb_checkpoints", "__pycache__"}


def leer_registros(datos, relativo):
    try:
        texto = datos.decode("utf-8-sig")
    except UnicodeDecodeError as error:
        incidencias.append({"archivo": relativo, "problema": "codificacion", "detalle": str(error)})
        return []
    if relativo.lower().endswith(".json"):
        try:
            valor = json.loads(texto)
            valores = valor if isinstance(valor, list) else [valor]
            return [(None, v) for v in valores]
        except json.JSONDecodeError as error:
            incidencias.append({"archivo": relativo, "problema": "json_invalido", "detalle": str(error)})
            return []
    salida = []
    for numero, linea in enumerate(texto.splitlines(), 1):
        if not linea.strip():
            continue
        try:
            salida.append((numero, json.loads(linea)))
        except json.JSONDecodeError as error:
            incidencias.append({"archivo": relativo, "linea": numero,
                                "problema": "linea_json_invalida", "detalle": str(error)})
    return salida


# Incluye JSONL y JSON de resultados individuales antiguos (por ejemplo Qwen2.5 7B).
# Ignora configuraciones y cualquier JSON que no contenga ejecuciones.
for archivo in sorted(RAIZ.rglob("*")):
    relativo_path = archivo.relative_to(RAIZ)
    if any(p in EXCLUIR for p in relativo_path.parts):
        continue
    if not archivo.is_file() or archivo.suffix.lower() not in {".json", ".jsonl"}:
        continue
    if archivo.name in comunes:
        continue
    if not archivo.resolve().is_relative_to(RAIZ):
        continue
    relativo = relativo_path.as_posix()
    datos = archivo.read_bytes()
    inicio_incidencias = len(incidencias)
    entradas = leer_registros(datos, relativo)
    reconocidos = 0
    for linea, registro in entradas:
        if not (isinstance(registro, dict) and "case_id" in registro
                and isinstance(registro.get("solicitud"), dict)):
            continue
        if not registro["solicitud"].get("model"):
            incidencias.append({"archivo": relativo, "linea": linea,
                                "problema": "ejecucion_sin_modelo"})
            continue
        reconocidos += 1
        digest = huella_bytes(serializar(registro).encode("utf-8"))
        execution_id = registro.get("execution_id")
        identidad = str(execution_id) if execution_id else "sin_id:" + digest
        if execution_id:
            id_a_huellas[identidad].add(digest)
        else:
            incidencias.append({"archivo": relativo, "linea": linea,
                                "problema": "execution_id_ausente", "case_id": registro["case_id"]})
        clave = (identidad, digest)
        origen = {"archivo": relativo, "linea": linea}
        if clave in registros:
            registros[clave]["origenes"].append(origen)
            copias_identicas += 1
        else:
            registros[clave] = {"registro": registro, "huella": digest,
                               "identidad": identidad, "origenes": [origen]}
    if reconocidos or (archivo.suffix.lower() == ".jsonl" and len(incidencias) > inicio_incidencias):
        archivos_bytes[relativo] = datos
        fuentes.append({"archivo": relativo, "sha256": huella_bytes(datos),
                        "bytes": len(datos), "apariciones_ejecuciones": reconocidos})

if not registros:
    raise ValueError("No se encontraron ejecuciones. Revisa RAIZ y las carpetas de modelos.")

for execution_id, huellas in id_a_huellas.items():
    if len(huellas) > 1:
        incidencias.append({"execution_id": execution_id, "problema": "mismo_id_con_contenidos_distintos",
                            "versiones": len(huellas)})

BLOQUE_THINK = re.compile(r"<think\b[^>]*>.*?</think\s*>", re.IGNORECASE | re.DOTALL)
ETIQUETA_THINK = re.compile(r"</?think\b", re.IGNORECASE)


def auditar_salida(respuesta):
    mensaje = respuesta.get("message", {}) or {}
    texto = mensaje.get("content", "") or ""
    if not isinstance(texto, str):
        texto = ""
    incrustado = bool(ETIQUETA_THINK.search(texto))
    final = BLOQUE_THINK.sub("", texto).strip()
    incompleto = bool(ETIQUETA_THINK.search(final))
    if incompleto:
        final = ""  # No exponer un bloque incompleto como si fuera la respuesta final.
    separado = bool(str(mensaje.get("thinking", "") or "").strip())
    modo = ("separado_e_incrustado" if separado and incrustado else
            "incrustado" if incrustado else "separado" if separado else "no_observado")
    reason = respuesta.get("done_reason")
    if reason == "length":
        tecnico = "truncada"
    elif reason == "stop" and final:
        tecnico = "terminada_con_texto_final"
    elif not respuesta or respuesta.get("error"):
        tecnico = "error_o_sin_respuesta"
    else:
        tecnico = "revisar_salida"
    return {"terminacion_auditada": tecnico, "modo_observado": modo,
            "bloque_incompleto": incompleto, "texto_final_recuperable": bool(final),
            "caracteres_respuesta_final": len(final)}


por_modelo = defaultdict(list)
por_perfil = defaultdict(list)
auditoria_registros = []

for item in registros.values():
    r = item["registro"]
    req = r["solicitud"]
    modelo = req["model"]
    caso_id = r["case_id"]
    respuesta = r.get("respuesta_original", {}) or {}
    grupo = r.get("grupo") or ("comparacion_inferida" if caso_id in ids_esperados else "sin_grupo")
    avisos = []
    for campo, esperado in (("sha256_casos", sha_casos), ("sha256_configuracion", sha_config)):
        if not r.get(campo):
            avisos.append(campo + "_no_registrado")
        elif r[campo] != esperado:
            avisos.append(campo + "_diferente")
    if not r.get("modelo_digest"):
        avisos.append("modelo_digest_no_registrado")
    if not r.get("ollama_version"):
        avisos.append("ollama_version_no_registrada")

    mensajes = req.get("messages", [])
    sistemas = [m.get("content") for m in mensajes if m.get("role") == "system"]
    usuarios = [m.get("content") for m in mensajes if m.get("role") == "user"]
    if sistemas != [config["system_prompt"]]:
        avisos.append("system_prompt_diferente_o_ausente")
    if caso_id in ids_esperados:
        esperado = config["user_prompt_template"].format(
            case_id=caso_id, contenido=casos_por_id[caso_id]["contenido"])
        if usuarios != [esperado] or len(mensajes) != 2:
            avisos.append("entrada_no_identica_al_caso_V3")

    salida = auditar_salida(respuesta)
    if req.get("think") is False and salida["modo_observado"] != "no_observado":
        avisos.append("think_false_con_razonamiento_observado")
    if r.get("estado") == "ok" and salida["modo_observado"] in {"incrustado", "separado_e_incrustado"}:
        avisos.append("estado_original_ok_con_bloque_incrustado")

    # Agrupar por solicitud efectiva, no solo por el nombre asignado a la configuración.
    protocolo = {
        "model": modelo, "modelo_digest": r.get("modelo_digest"),
        "ollama_version": r.get("ollama_version"),
        "solicitud_sin_mensajes": {k: v for k, v in req.items() if k != "messages"},
        "system_messages": sistemas,
    }
    perfil = huella_bytes(serializar(protocolo).encode("utf-8"))[:16]
    linea = {
        "execution_id": r.get("execution_id"), "huella_registro": item["huella"],
        "origenes": item["origenes"], "modelo": modelo, "case_id": caso_id,
        "grupo": grupo, "estado_original": r.get("estado"),
        "configuracion_declarada": r.get("configuracion_modelo"),
        "perfil_solicitud": perfil, "think_solicitado": req.get("think", "omitido"),
        "options": req.get("options", {}), "reintentos_declarados": r.get("reintentos"),
        "tiempo_cliente_s": r.get("tiempo_cliente_s"),
        "tokens_generados_totales": respuesta.get("eval_count"),
        "load_duration_ns": respuesta.get("load_duration"),
        "prompt_eval_duration_ns": respuesta.get("prompt_eval_duration"),
        "eval_duration_ns": respuesta.get("eval_duration"),
        "prompt_eval_cached_count": respuesta.get("prompt_eval_cached_count"),
        **salida, "avisos": avisos,
    }
    auditoria_registros.append(linea)
    por_modelo[modelo].append(linea)
    por_perfil[(modelo, perfil)].append(linea)

resumen_modelos = []
for modelo, rr in sorted(por_modelo.items()):
    comparaciones = [r for r in rr if r["case_id"] in ids_esperados
                     and r["grupo"] in ("comparacion", "comparacion_inferida")]
    vistos = {r["case_id"] for r in comparaciones}
    con_texto = {r["case_id"] for r in comparaciones
                if r["terminacion_auditada"] == "terminada_con_texto_final"}
    resumen_modelos.append({
        "modelo": modelo, "registros_distintos": len(rr),
        "casos_vistos": sorted(vistos), "casos_faltantes": sorted(ids_esperados - vistos),
        "casos_con_algun_intento_completo": sorted(con_texto),
        "perfiles_solicitud": len({r["perfil_solicitud"] for r in rr}),
    })

resumen_perfiles = []
for (modelo, perfil), rr in sorted(por_perfil.items()):
    comparaciones = [r for r in rr if r["case_id"] in ids_esperados
                     and r["grupo"] in ("comparacion", "comparacion_inferida")]
    resumen_perfiles.append({
        "modelo": modelo, "perfil_solicitud": perfil,
        "configuraciones_declaradas": sorted({r["configuracion_declarada"] for r in rr
                                              if r["configuracion_declarada"]}),
        "think_solicitado": rr[0]["think_solicitado"], "options": rr[0]["options"],
        "casos_vistos_comparacion": sorted({r["case_id"] for r in comparaciones}),
        "intentos_por_caso": dict(Counter(r["case_id"] for r in comparaciones)),
        "modos_observados": dict(Counter(r["modo_observado"] for r in rr)),
        "registros_distintos": len(rr),
    })

faltantes = sorted(set(MODELOS_ESPERADOS) - set(por_modelo))
extras = sorted(set(por_modelo) - set(MODELOS_ESPERADOS))
auditoria = {
    "version_auditoria": "1.0", "creado_utc": datetime.now(timezone.utc).isoformat(),
    "alcance": "Inventario de ejecuciones; no es una evaluación de calidad ni un ranking.",
    "notas": [
        "Los archivos originales se conservan byte a byte en fuentes/.",
        "Los estados originales no se modifican. La auditoría añade observaciones independientes.",
        "Las copias idénticas no se cuentan dos veces; reintentos y rescates se conservan.",
        "Un caso completo en alguna configuración no demuestra seis casos completos en todas las configuraciones.",
        "Campos ausentes de registros antiguos se marcan como ausentes; no se inventan.",
        "No se calcula una nota automática de fidelidad, calidad o adecuación al español.",
        "No observar razonamiento en campos o etiquetas no demuestra ausencia de procesamiento interno.",
    ],
    "sha256_casos": sha_casos, "sha256_configuracion": sha_config,
    "modelos_esperados": MODELOS_ESPERADOS, "modelos_faltantes": faltantes,
    "modelos_adicionales": extras, "numero_registros_distintos": len(registros),
    "copias_identicas_no_recontadas": copias_identicas,
    "fuentes": fuentes, "incidencias_lectura_identidad": incidencias,
    "resumen_modelos": resumen_modelos, "resumen_perfiles": resumen_perfiles,
    "auditoria_registros": auditoria_registros,
}

# Un único archivo derivado. No contiene modelos, CSV de tesis ni notebooks.
SALIDA.mkdir(exist_ok=True)
archivo_zip = SALIDA / NOMBRE_SALIDA
temporal = SALIDA / (NOMBRE_SALIDA + ".tmp")
try:
    with zipfile.ZipFile(temporal, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for nombre, datos in comunes.items():
            z.writestr(nombre, datos)
        for nombre, datos in archivos_bytes.items():
            z.writestr("fuentes/" + nombre, datos)
        z.writestr("auditoria.json", json.dumps(auditoria, ensure_ascii=False, indent=2))
    with zipfile.ZipFile(temporal) as z:
        error_zip = z.testzip()
        if error_zip is not None:
            raise ValueError("Error de integridad en el ZIP: " + error_zip)
    temporal.replace(archivo_zip)
finally:
    if temporal.exists():
        temporal.unlink()

print("Raíz revisada:", RAIZ)
print("Modelos del plan encontrados:", len(set(por_modelo) & set(MODELOS_ESPERADOS)), "/ 22")
print("Fuentes incluidas:", len(fuentes))
print("Registros distintos:", len(registros))
print("Copias idénticas no recontadas:", copias_identicas)
print("Incidencias de lectura/identidad:", len(incidencias))
print("\nCobertura por modelo (no equivale a calidad aprobada):")
for fila in resumen_modelos:
    print("-", fila["modelo"], "| casos:", str(len(fila["casos_vistos"])) + "/6",
          "| perfiles:", fila["perfiles_solicitud"],
          "| registros:", fila["registros_distintos"])
if faltantes:
    print("\nMODELOS SIN ARCHIVO DE RESULTADOS IDENTIFICADO:")
    print("\n".join(faltantes))
if extras:
    print("\nModelos adicionales:", ", ".join(extras))
if incidencias:
    print("\nHay incidencias. Se conservaron en auditoria.json dentro del ZIP.")
print("\nArchivo creado:", archivo_zip)
print("No se ejecutó ninguna inferencia ni se modificaron los resultados originales.")


Raíz revisada: C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\05_Pruebas
Modelos del plan encontrados: 22 / 22
Fuentes incluidas: 31
Registros distintos: 209
Copias idénticas no recontadas: 6
Incidencias de lectura/identidad: 0

Cobertura por modelo (no equivale a calidad aprobada):
- command-r7b:7b-12-2024-q4_K_M | casos: 6/6 | perfiles: 1 | registros: 7
- deepseek-r1:8b-0528-qwen3-q4_K_M | casos: 6/6 | perfiles: 2 | registros: 8
- gemma4:12b-it-q4_K_M | casos: 6/6 | perfiles: 2 | registros: 14
- gemma4:26b-a4b-it-qat | casos: 6/6 | perfiles: 2 | registros: 14
- gemma4:31b-it-qat | casos: 6/6 | perfiles: 1 | registros: 7
- gpt-oss:20b | casos: 6/6 | perfiles: 2 | registros: 8
- granite4.2:30b-q4_K_M | casos: 6/6 | perfiles: 1 | registros: 7
- granite4.2:3b-q4_K_M | casos: 6/6 | perfiles: 2 | registros: 14
- granite4.2:8b-q4_K_M | casos: 6/6 | perfiles: 2 | registros: 14
- lfm2.5:8b-a1b-q4_K_M | casos: 6/6 | perfiles: 1 | registros: 7
- llama3.1:8b-instruct-q4_K_M | casos: 6/6 | per